In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

# 17 · Making a chocolate bar bend 🍫

We opened the course with a chocolate bar we could only *look* at. Let's finish by
making it **bend**. **Linear elasticity** is our first genuinely
**vector-valued** PDE: the unknown is a displacement $\mathbf{u}(x)$ that moves
every material point. Clamp the bar at one end, let gravity pull, and solve for
the sag.

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

# The very chocolate bar we built in notebook 2 — a ridged solid
# (clamped at the x=0 end, hanging out into the room).
def chocolate_bar():
    n_peaks, b, depth, base_h, ph, r_fil = 3, 2.0, 3.0, 0.4, 2.6, 0.2
    W = n_peaks*b
    def trap(a, b, c, d):
        return Face(Wire([Segment(a, b), Segment(b, c), Segment(c, d), Segment(d, a)]))
    def peak(cx, cy, z0, bx, by, h):
        rx = Prism(trap(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx+bx/2, cy-by/2, z0),
                        Pnt(cx+0.1*bx, cy-by/2, z0+h), Pnt(cx-0.1*bx, cy-by/2, z0+h)), Vec(0, by, 0))
        ry = Prism(trap(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx-bx/2, cy+by/2, z0),
                        Pnt(cx-bx/2, cy+0.1*by, z0+h), Pnt(cx-bx/2, cy-0.1*by, z0+h)), Vec(bx, 0, 0))
        return rx * ry
    solid = Box(Pnt(0, -0.3, 0), Pnt(W, depth + 0.3, base_h))
    for i in range(n_peaks):
        solid = solid + peak((i + 0.5)*b, depth/2, base_h, b, depth + 0.6, ph)
    along_y = lambda e: abs(e.vertices[0].p[1]-e.vertices[1].p[1]) > abs(e.vertices[0].p[0]-e.vertices[1].p[0])
    valleys = [e for e in solid.edges
               if abs(e.center[2]-base_h) < 0.05 and along_y(e) and 0.1 < e.center[0] < W-0.1]
    solid = solid.MakeFillet(valleys, r_fil)
    return solid * Box(Pnt(-1, 0, -1), Pnt(W + 1, depth, 10*ph)), W, depth

bar, L, depth = chocolate_bar()
bar.faces.Min(X).name = "clamp"                         # the wall-mounted end
mesh = Mesh(OCCGeometry(bar).GenerateMesh(maxh=1.0))
mesh.Curve(2)
Draw(mesh)

## 1. Material law: stress from strain

A small displacement produces a **strain** $\varepsilon(\mathbf u)=\tfrac12(\nabla
\mathbf u+\nabla\mathbf u^\top)$, and Hooke's law turns that into a **stress**
$\sigma = 2\mu\,\varepsilon + \lambda\,\mathrm{tr}(\varepsilon)\,I$. The two
**Lamé parameters** $\mu,\lambda$ come from Young's modulus $E$ and Poisson's
ratio $\nu$. `Sym`, `Trace` and `Id` build the tensors symbolically.

In [ ]:
E, nu = 1.0e4, 0.3
mu  = E / (2*(1+nu))
lam = E*nu / ((1+nu)*(1-2*nu))

def strain(u): return Sym(Grad(u))
def stress(u): return 2*mu*strain(u) + lam*Trace(strain(u))*Id(3)

## 2. The weak form and the solve

The variational problem is $\int_\Omega \sigma(\mathbf u):\varepsilon(\mathbf v)
\,dx = \int_\Omega \mathbf f\cdot\mathbf v\,dx$, with a body force $\mathbf f$
(gravity, pulling in $-z$) and the clamp imposed as a Dirichlet condition on the
`VectorH1` space. The system is symmetric positive definite, so our trusty
`sparsecholesky` handles it directly.

In [ ]:
fes = VectorH1(mesh, order=2, dirichlet="clamp")
u, v = fes.TnT()
a = BilinearForm(InnerProduct(stress(u), strain(v))*dx).Assemble()
f = LinearForm(CF((0, 0, -1.5))*v*dx).Assemble()        # gravity body force

gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec

tip = gfu(mesh(L - 0.05, depth/2, 0.3))
print(f"tip displacement (x,y,z) = ({tip[0]:+.3f}, {tip[1]:+.3f}, {tip[2]:+.3f})")
print(f"the free end sags {abs(tip[2]):.2f} units downward")

## 3. See it bend

`Draw(..., deformation=gfu)` warps the mesh by the displacement so the sag is
visible. We colour it by the **von Mises stress** — the scalar that engineers
compare against a material's yield strength. It peaks, as intuition demands, at
the **clamped top edge**, where the bar is worked hardest.

In [ ]:
dev = stress(gfu) - (Trace(stress(gfu))/3)*Id(3)
vonMises = sqrt(3/2 * InnerProduct(dev, dev))
Draw(vonMises, mesh, "von Mises stress", deformation=gfu)

## 🍫 → ☕ → 🍫

The chocolate bar that opened the course now bends under load — vectors instead of
scalars, but the very same workflow. One last, **advanced** glimpse remains:
letting the coffee couple temperature and flow and stir *itself*.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("15-outlook-unfitted", "15 · Outlook — unfitted FEM with ngsxfem 🫧")
    _next = ("18-buoyant-convection", "18 · A glimpse of more: buoyant convection ☕🔥")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))